In [47]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import RidgeClassifier, RidgeClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder,MinMaxScaler
from sklearn.metrics import f1_score, accuracy_score

In [48]:
# Load Data 
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

train_df.head()


,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [49]:
# now that we have the data imported we should start by first getting all the unique types of vals in non int cols
gender_col            = train_df['gender']
marital_status_col    = train_df['marital_status']
educational_level_col = train_df['education_level']
employment_status_col = train_df['employment_status']
loan_purposes_col     = train_df['loan_purpose']
grade_subgrade_col    = train_df['grade_subgrade']

print(
    'Here are the following unique values in string cols\n',
    f'Gender Col: {gender_col.unique()}\n',
    f'Marital Status Col: {marital_status_col.unique()}\n',
    f'Educational Level Col: {educational_level_col.unique()}\n',
    f'Employment Status Col: {employment_status_col.unique()}\n',
    f'Loan Purposes Col: {loan_purposes_col.unique()}\n',
    f'Grade subgrade Col: {grade_subgrade_col.unique()}'
)

Here are the following unique values in string cols
 Gender Col: ['Female' 'Male' 'Other']
 Marital Status Col: ['Single' 'Married' 'Divorced' 'Widowed']
 Educational Level Col: ['High School' "Master's" "Bachelor's" 'PhD' 'Other']
 Employment Status Col: ['Self-employed' 'Employed' 'Unemployed' 'Retired' 'Student']
 Loan Purposes Col: ['Other' 'Debt consolidation' 'Home' 'Education' 'Vacation' 'Car'
 'Medical' 'Business']
 Grade subgrade Col: ['C3' 'D3' 'C5' 'F1' 'D1' 'D5' 'C2' 'C1' 'F5' 'D4' 'C4' 'D2' 'E5' 'B1'
 'B2' 'F4' 'A4' 'E1' 'F2' 'B4' 'E4' 'B3' 'E3' 'B5' 'E2' 'F3' 'A5' 'A3'
 'A1' 'A2']


In [50]:
# now we prepare our custom ranking for ordinal cols
grade_subgrade_ordered = [
    'F5', 'F4', 'F3', 'F2', 'F1',  
    'E5', 'E4', 'E3', 'E2', 'E1',  
    'D5', 'D4', 'D3', 'D2', 'D1',  
    'C5', 'C4', 'C3', 'C2', 'C1',  
    'B5', 'B4', 'B3', 'B2', 'B1',  
    'A5', 'A4', 'A3', 'A2', 'A1'   
]

educational_level_ordered = [
    'Other', 'High School', "Bachelor's", "Master's", 'PhD'
]

In [51]:
# here we intiliaze and fit the encoder
ordinal_categories = [
    grade_subgrade_ordered,
    educational_level_ordered
]

ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories
)

In [52]:
ordinal_cols = ['grade_subgrade', 'education_level']
ordinal_encoder.fit(train_df[ordinal_cols])
ordinal_encoded_array = ordinal_encoder.transform(train_df[ordinal_cols])

ordinal_encoded_df = pd.DataFrame(
    ordinal_encoded_array, 
    columns=ordinal_cols, 
    index=train_df.index
)
train_df[ordinal_cols[0]] = ordinal_encoded_df[ordinal_cols[0]]
train_df[ordinal_cols[1]] = ordinal_encoded_df[ordinal_cols[1]]

train_df


,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,1.0,Self-employed,Other,17.0,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,3.0,Employed,Debt consolidation,12.0,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,1.0,Employed,Debt consolidation,15.0,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,1.0,Employed,Debt consolidation,4.0,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,1.0,Employed,Other,14.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
593989,593989,23004.26,0.152,703,20958.37,10.92,Female,Single,1.0,Employed,Business,17.0,1.0
593990,593990,35289.43,0.105,559,3257.24,14.62,Male,Single,2.0,Employed,Debt consolidation,0.0,1.0
593991,593991,47112.64,0.072,675,929.27,14.13,Female,Married,2.0,Employed,Debt consolidation,19.0,1.0
593992,593992,76748.44,0.067,740,16290.40,9.87,Male,Single,2.0,Employed,Debt consolidation,23.0,1.0


In [53]:
nominal_cols = ['marital_status', 'employment_status', 'loan_purpose', 'gender']
oneHot = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False) 
hot_encoded_array = oneHot.fit_transform(train_df[nominal_cols])
new_ohe_cols = oneHot.get_feature_names_out(nominal_cols)
hot_encoded_df = pd.DataFrame(
    hot_encoded_array, 
    columns=new_ohe_cols, 
    index=train_df.index
)

train_df = pd.concat([train_df.drop(columns=nominal_cols), hot_encoded_df], axis=1)
train_df.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,education_level,grade_subgrade,loan_paid_back,marital_status_Married,...,employment_status_Unemployed,loan_purpose_Car,loan_purpose_Debt consolidation,loan_purpose_Education,loan_purpose_Home,loan_purpose_Medical,loan_purpose_Other,loan_purpose_Vacation,gender_Male,gender_Other
0,0,29367.99,0.084,736,2528.42,13.67,1.0,17.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,1,22108.02,0.166,636,4593.10,12.92,3.0,12.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,2,49566.20,0.097,694,17005.15,9.76,1.0,15.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,3,46858.25,0.065,533,4682.48,16.10,1.0,4.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,25496.70,0.053,665,12184.43,10.21,1.0,14.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [54]:
# init scalers 
minmaxScaler   = MinMaxScaler()
scalable_cols = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
scaled_arr = minmaxScaler.fit_transform(train_df[scalable_cols])
scaled_df = pd.DataFrame(
    scaled_arr,
    columns=scalable_cols,
    index=train_df.index
)
train_df = pd.concat([train_df.drop(columns=scalable_cols), scaled_df], axis=1)

In [55]:
drop_cols = ['id', 'loan_paid_back']
X = train_df.drop(drop_cols, axis=1)
y = train_df['loan_paid_back']

In [ ]:
# init models 